# Week1_ex3 — Power Factor Correction with a Shunt Capacitor

Reproduces Example 2.3 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): a 100 kW, 0.8 power factor load fed over a lossless or a lossy line, with a switchable capacitor at the load bus. It is the first case with a shunt element, and the capacitor's output turns out to depend on the voltage it sits at.

![PowerWorld one-line](Images/Week1_ex3_result_image.png)

In [1]:
import numpy as np
import pandas as pd
import pandapower as pp

# 1. Per-unit bases used by the PowerWorld case (Simulator Options > Power Flow Solution > General)
S_BASE_KVA = 100.0                          # this case uses a kVA base, so every table is in kW / kvar
V_BASE_KV = 16.0
Z_BASE_OHM = V_BASE_KV**2 / (S_BASE_KVA / 1000)   # 2560 ohm
KVA_TOL = 0.1                               # kVA convergence tolerance (Common Options)

# 2. Data read once from PowerWorld Simulator (Run Mode, Model Explorer > Network > Branches Input / Loads / Buses)
LINES = {"Lossless Line": (0.0, 0.00001), "Line Including Losses": (0.05, 0.10)}   # R, X in pu
LOAD_KW, LOAD_KVAR = 100.10, 75.08          # Loads table; the one-line rounds these to 100 / 75
SLACK_ANGLE_DEG = -0.23                     # Buses table, Bus A
CAP_STEP_KVAR = 20.0                        # one click on the one-line arrow, rated at 1.0 pu

pd.DataFrame({
    "Quantity": ["System base", "Z_base", "Lossless line R / X", "Lossy line R / X", "Load", "Capacitor step"],
    "PowerWorld": ["100 kVA", "-", "0 / 0.00001 pu", "0.05 / 0.10 pu", "100.10 kW + 75.08 kvar", "20 kvar"],
    "pandapower": ["sn_mva=0.1", f"{Z_BASE_OHM:.0f} ohm", "in_service toggled per case", "ohm_per_km = pu * Z_base, 1 km",
                   "p_mw=0.1001, q_mvar=0.07508", "create_shunt(q_mvar=-0.02, step=n)"],
})

,Quantity,PowerWorld,pandapower
0,System base,100 kVA,sn_mva=0.1
1,Z_base,-,2560 ohm
2,Lossless line R / X,0 / 0.00001 pu,in_service toggled per case
3,Lossy line R / X,0.05 / 0.10 pu,"ohm_per_km = pu * Z_base, 1 km"
4,Load,100.10 kW + 75.08 kvar,"p_mw=0.1001, q_mvar=0.07508"
5,Capacitor step,20 kvar,"create_shunt(q_mvar=-0.02, step=n)"


In [2]:
# 3. One network with both lines; the breakers on the one-line become in_service flags
def build_case(line_closed, cap_steps):
    net = pp.create_empty_network(f_hz=60, sn_mva=S_BASE_KVA / 1000)
    bus_a = pp.create_bus(net, V_BASE_KV, name="Bus A")
    bus_b = pp.create_bus(net, V_BASE_KV, name="Load Bus")
    pp.create_ext_grid(net, bus_a, vm_pu=1.0, va_degree=SLACK_ANGLE_DEG)
    for name, (r, x) in LINES.items():
        pp.create_line_from_parameters(net, bus_a, bus_b, length_km=1.0, r_ohm_per_km=r * Z_BASE_OHM,
                                       x_ohm_per_km=x * Z_BASE_OHM, c_nf_per_km=0.0, max_i_ka=1.0,
                                       name=name, in_service=(name == line_closed))
    pp.create_load(net, bus_b, p_mw=LOAD_KW / 1000, q_mvar=LOAD_KVAR / 1000)
    # pandapower counts reactive power consumed by the shunt as positive, so a capacitor is negative.
    # q_mvar is the output per step at the shunt's rated voltage, not what it actually delivers.
    pp.create_shunt(net, bus_b, q_mvar=-CAP_STEP_KVAR / 1000, p_mw=0.0, step=cap_steps, max_step=10, name="Capacitor")
    pp.runpp(net, numba=False)
    return net


def summary(net):
    gen = net.res_ext_grid.iloc[0]
    return {
        "Load Bus voltage [kV]": net.res_bus.vm_pu.iat[1] * V_BASE_KV,
        "Gen P [kW]": gen.p_mw * 1000, "Gen Q [kvar]": gen.q_mvar * 1000,
        "Gen S [kVA]": np.hypot(gen.p_mw, gen.q_mvar) * 1000,
        "Line P loss [kW]": net.res_line.pl_mw.sum() * 1000, "Line Q loss [kvar]": net.res_line.ql_mvar.sum() * 1000,
        "Capacitor output [kvar]": -net.res_shunt.q_mvar.iat[0] * 1000,
    }


cases = {
    "Lossless, cap 0": ("Lossless Line", 0),
    "Lossless, cap 80": ("Lossless Line", 4),
    "Lossy, cap 0": ("Line Including Losses", 0),
    "Lossy, cap 80": ("Line Including Losses", 4),
    "Lossy, cap 100": ("Line Including Losses", 5),
}
results = {label: summary(build_case(*args)) for label, args in cases.items()}
pd.DataFrame(results).round(3)

,"Lossless, cap 0","Lossless, cap 80","Lossy, cap 0","Lossy, cap 80","Lossy, cap 100"
Load Bus voltage [kV],16.000,16.000,13.601,14.971,15.344
Gen P [kW],100.100,100.100,110.933,105.837,105.703
Gen Q [kvar],75.082,-4.919,96.746,16.516,-5.681
Gen S [kVA],125.129,100.221,147.193,107.118,105.855
Line P loss [kW],0.000,0.000,10.833,5.737,5.603
Line Q loss [kvar],0.002,0.001,21.666,11.474,11.205
Capacitor output [kvar],0.000,80.000,0.000,70.038,91.967


In [3]:
# 4. PowerWorld values read once from PowerWorld Simulator (Run Mode); the one-line rounds Gen values to whole kW / kvar
H = 0.5   # half of a whole-number display digit
pw = [
    ("Lossless, cap 0", "Gen P [kW]", 100.10, KVA_TOL, "Generators table"),
    ("Lossless, cap 0", "Gen Q [kvar]", 75.08, KVA_TOL, "Generators table"),
    ("Lossless, cap 80", "Gen Q [kvar]", -5.0, H, "One-line"),
    ("Lossless, cap 80", "Gen S [kVA]", 100.0, H, "One-line"),
    ("Lossless, cap 80", "Capacitor output [kvar]", 80.0, 0.05, "One-line"),
    ("Lossy, cap 0", "Load Bus voltage [kV]", 13.60, 5e-3, "One-line"),
    ("Lossy, cap 0", "Gen P [kW]", 111.0, H, "One-line"),
    ("Lossy, cap 0", "Gen Q [kvar]", 97.0, H, "One-line"),
    ("Lossy, cap 0", "Gen S [kVA]", 147.0, H, "One-line"),
    ("Lossy, cap 0", "Line P loss [kW]", 10.81, KVA_TOL, "One-line"),
    ("Lossy, cap 0", "Line Q loss [kvar]", 21.62, KVA_TOL, "One-line"),
    ("Lossy, cap 80", "Load Bus voltage [kV]", 14.97, 5e-3, "One-line"),
    ("Lossy, cap 80", "Gen P [kW]", 106.0, H, "One-line"),
    ("Lossy, cap 80", "Gen Q [kvar]", 17.0, H, "One-line"),
    ("Lossy, cap 80", "Gen S [kVA]", 107.0, H, "One-line"),
    ("Lossy, cap 80", "Line P loss [kW]", 5.73, KVA_TOL, "One-line"),
    ("Lossy, cap 80", "Line Q loss [kvar]", 11.46, KVA_TOL, "One-line"),
    ("Lossy, cap 80", "Capacitor output [kvar]", 70.0, 0.05, "One-line"),
    ("Lossy, cap 100", "Capacitor output [kvar]", 92.0, H, "One-line"),
]
cmp = pd.DataFrame(pw, columns=["Case", "Quantity", "PowerWorld", "Tolerance", "Source"])
cmp.insert(3, "pandapower", [results[c][q] for c, q in zip(cmp["Case"], cmp["Quantity"])])
cmp["Abs error"] = (cmp["pandapower"] - cmp["PowerWorld"]).abs()
assert (cmp["Abs error"] <= cmp["Tolerance"]).all(), cmp[cmp["Abs error"] > cmp["Tolerance"]]
cmp.round({"pandapower": 3, "Abs error": 3})

,Case,Quantity,PowerWorld,pandapower,Tolerance,Source,Abs error
0,"Lossless, cap 0",Gen P [kW],100.10,100.100,0.100,Generators table,0.000
1,"Lossless, cap 0",Gen Q [kvar],75.08,75.082,0.100,Generators table,0.002
2,"Lossless, cap 80",Gen Q [kvar],-5.00,-4.919,0.500,One-line,0.081
3,"Lossless, cap 80",Gen S [kVA],100.00,100.221,0.500,One-line,0.221
4,"Lossless, cap 80",Capacitor output [kvar],80.00,80.000,0.050,One-line,0.000
5,"Lossy, cap 0",Load Bus voltage [kV],13.60,13.601,0.005,One-line,0.001
6,"Lossy, cap 0",Gen P [kW],111.00,110.933,0.500,One-line,0.067
7,"Lossy, cap 0",Gen Q [kvar],97.00,96.746,0.500,One-line,0.254
8,"Lossy, cap 0",Gen S [kVA],147.00,147.193,0.500,One-line,0.193
9,"Lossy, cap 0",Line P loss [kW],10.81,10.833,0.100,One-line,0.023


In [4]:
# 5. A capacitor is a constant impedance: what it delivers scales with V^2, so it weakens exactly when voltage sags
rows = []
for steps in range(0, 6):
    net = build_case("Line Including Losses", steps)
    vm = net.res_bus.vm_pu.iat[1]
    actual = -net.res_shunt.q_mvar.iat[0] * 1000
    rows.append({"Steps": steps, "Rated output [kvar]": steps * CAP_STEP_KVAR, "Load Bus voltage [pu]": vm,
                 "Actual output [kvar]": actual, "Actual / rated": actual / (steps * CAP_STEP_KVAR) if steps else np.nan,
                 "V^2 [pu]": vm**2, "Line P loss [kW]": net.res_line.pl_mw.sum() * 1000})
law = pd.DataFrame(rows)
assert (law["Actual / rated"] - law["V^2 [pu]"]).abs().max() < 1e-9
law.round(4)

,Steps,Rated output [kvar],Load Bus voltage [pu],Actual output [kvar],Actual / rated,V^2 [pu],Line P loss [kW]
0,0,0.0,0.8501,0.0000,NaN,0.7227,10.8329
1,1,20.0,0.8704,15.1517,0.7576,0.7576,8.9834
2,2,40.0,0.8914,31.7838,0.7946,0.7946,7.4847
3,3,60.0,0.9131,50.0303,0.8338,0.8338,6.3846
4,4,80.0,0.9357,70.0378,0.8755,0.8755,5.7371
5,5,100.0,0.9590,91.9667,0.9197,0.9197,5.6027


In [5]:
# 6. Pitfall: a positive q_mvar is a reactor, not a capacitor -- the case still converges, just to the wrong answer
net_cap = build_case("Line Including Losses", 4)
net_wrong = build_case("Line Including Losses", 4)
net_wrong.shunt.q_mvar = +CAP_STEP_KVAR / 1000
pp.runpp(net_wrong, numba=False)

pd.DataFrame({
    "Shunt q_mvar per step": [-CAP_STEP_KVAR / 1000, +CAP_STEP_KVAR / 1000],
    "Behaves as": ["capacitor", "reactor"],
    "Load Bus voltage [kV]": [net_cap.res_bus.vm_pu.iat[1] * V_BASE_KV, net_wrong.res_bus.vm_pu.iat[1] * V_BASE_KV],
    "Line P loss [kW]": [net_cap.res_line.pl_mw.sum() * 1000, net_wrong.res_line.pl_mw.sum() * 1000],
}).round(3)

,Shunt q_mvar per step,Behaves as,Load Bus voltage [kV],Line P loss [kW]
0,-0.02,capacitor,14.971,5.737
1,0.02,reactor,12.406,20.952
